<h1 style="text-align: center; font-family: Arial; font-weight: bold; color:green; font-size:36px;">Renewable Energy Forecasting
    Dashboard for the Republic of Ireland</h1>

<b>Author:</b> J.A Montuya<br />
<b>Student ID:</b> 2025040<br />


In [42]:
# Import Libraries
import dash
from dash import dcc, html, Dash
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt


# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import pickle
import webbrowser

In [43]:
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Load Datasets

In [45]:
# Import Datasets

# Electricity Prices per Country (EU)
eu_prices_df = pd.read_csv('./Datasets/electricity-prices-by-usertype-eu.csv')

# Indigenous Energy Production in Ireland
ie_power_production_df= pd.read_csv('./Datasets/AA.csv')

In [46]:
eu_prices_df.head()

,DATAFLOW,LAST UPDATE,freq,product,currency,unit,indic_en,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TEN00117(1.0),18/04/25 23:00:00,A,6000,EUR,KWH,MSHH,AL,2013,0.1156,NaN,NaN
1,ESTAT:TEN00117(1.0),18/04/25 23:00:00,A,6000,EUR,KWH,MSHH,AL,2014,0.1156,NaN,NaN
2,ESTAT:TEN00117(1.0),18/04/25 23:00:00,A,6000,EUR,KWH,MSHH,AL,2015,0.0812,NaN,NaN
3,ESTAT:TEN00117(1.0),18/04/25 23:00:00,A,6000,EUR,KWH,MSHH,AL,2016,0.0824,NaN,NaN
4,ESTAT:TEN00117(1.0),18/04/25 23:00:00,A,6000,EUR,KWH,MSHH,AL,2017,0.0844,NaN,NaN


In [47]:
ie_power_production_df.head()

,Year of Period,Month of Period,Unit,Coal,Combust. Renew.,Hydro,Natural Gas,Oil,Other,Peat & BM,Solar Farms,Wastes,Wind
0,2025,January,ktoe,8.8,1.1,7.5,125.0,3.3,0.1,4.7,2.5,4.6,94.7
1,2024,January,ktoe,8.6,1.1,11.3,113.8,2.4,0.1,3.4,1.5,4.8,98.2
2,2024,February,ktoe,7.4,1.0,9.8,85.3,1.0,0.1,4.6,1.7,4.3,103.0
3,2024,March,ktoe,7.5,1.2,10.0,87.7,0.5,0.1,4.2,3.0,1.9,114.4
4,2024,April,ktoe,7.1,1.1,7.6,92.2,0.4,0.1,4.4,5.6,3.3,79.9


## Data Preparation

### 1. Electricity Prices in EU in Euros per KWH

In [50]:
# Country Code Mapping
country_code_dict = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CY": "Cyprus",
    "CZ": "Czech_Republic",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "EL": "Greece",
    "ES": "Spain",
    "EU27_2020": "European_Union",
    "FI": "Finland",
    "FR": "France",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia"
}

# Select usable columns
eu_prices_df=eu_prices_df[['TIME_PERIOD','geo', 'unit', 'OBS_VALUE','currency']]

# Add feature countries by mapping country code to country name 
eu_prices_df['country'] = eu_prices_df['geo'].map(country_code_dict)

# Remove empty rows
eu_prices_df.dropna(axis=0, inplace=True)

# Drop Duplicate values
eu_prices_df = eu_prices_df.drop_duplicates(subset=['country', 'TIME_PERIOD'])

# Restructure dataframe to make years as columns and countries to records
eu_prices_pivot_df = eu_prices_df[['country','TIME_PERIOD','OBS_VALUE']].pivot(index='country', columns='TIME_PERIOD', values='OBS_VALUE')

# Sort Values
eu_prices_pivot_df = eu_prices_pivot_df.sort_index()

# Show 5 records
eu_prices_pivot_df.head()

TIME_PERIOD,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
country,,,,,,,,,,,,
Austria,0.2082,0.2021,0.2009,0.2034,0.1950,0.1966,0.2034,0.2111,0.2216,0.2249,0.2653,0.2731
Belgium,0.2173,0.2097,0.2126,0.2544,0.2857,0.2824,0.2839,0.2792,0.2702,0.3437,0.4350,0.3354
Bulgaria,0.0924,0.0832,0.0942,0.0956,0.0955,0.0979,0.0997,0.0997,0.1024,0.1093,0.1138,0.1187
Croatia,0.1372,0.1312,0.1317,0.1311,0.1196,0.1311,0.1321,0.1301,0.1291,0.1354,0.1480,0.1472
Cyprus,0.2760,0.2291,0.1957,0.1527,0.1863,0.1893,0.2203,0.2133,0.1976,0.2607,0.3739,0.3241


In [51]:
# Rename Columns
ie_power_production_df.rename(columns={
    'Year of Period':'Year',
    'Month of Period':'Month',
    'Natural Gas': 'Natural_Gas',
    'Peat & BM': 'Peat_&_BM',
    'Solar Farms': 'Solar_Farms'
}, inplace=True)

# Classify Columns to Renewable and Non-Renewable Sources
renewable_energy = ['Hydro', 'Solar_Farms', 'Wastes', 'Wind']
nonrenewable_energy = ['Coal', 'Natural_Gas', 'Oil', 'Peat_&_BM', 'Other']


# Sum all renewable energy sources
ie_power_production_df['Renewable_Energy'] = ie_power_production_df[renewable_energy].sum(axis=1)

# Sum all non-renewable energy sources
ie_power_production_df['Non_Renewable_Energy'] = ie_power_production_df[nonrenewable_energy].sum(axis=1)

# Transform Months to Numeric
ie_power_production_df['Month_Numeric'] =  pd.to_datetime(ie_power_production_df['Month'], format='%B').dt.month

# Transform Year and Month to Date
ie_power_production_df['Date'] = pd.to_datetime(ie_power_production_df['Year'].astype(str) + '-' + ie_power_production_df['Month_Numeric'].astype(str).str.zfill(2) + '-01')

# Check dataset if changes have been made 
ie_power_production_df.head()

# Set Date as Index
ie_power_production_df.set_index('Date', inplace=True)

# Sort Date in ascending order
ie_power_production_df.sort_index(inplace=True)

ie_power_production_df.head()

,Year,Month,Unit,Coal,Combust. Renew.,Hydro,Natural_Gas,Oil,Other,Peat_&_BM,Solar_Farms,Wastes,Wind,Renewable_Energy,Non_Renewable_Energy,Month_Numeric
Date,,,,,,,,,,,,,,,,
2010-01-01,2010,January,ktoe,34.4,0.0,8.3,126.8,6.7,1.7,16.7,0.0,0.0,21.3,29.6,186.3,1
2010-02-01,2010,February,ktoe,35.0,0.0,5.1,120.0,2.5,1.9,16.3,0.0,0.0,13.0,18.1,175.7,2
2010-03-01,2010,March,ktoe,26.5,0.0,2.5,136.8,1.4,1.9,14.8,0.0,0.0,21.6,24.1,181.4,3
2010-04-01,2010,April,ktoe,20.0,0.0,6.3,117.6,1.2,2.0,18.3,0.0,0.0,16.0,22.3,159.1,4
2010-05-01,2010,May,ktoe,23.3,0.0,1.1,116.4,0.6,1.6,15.2,0.0,0.0,14.5,15.6,157.1,5


## Data Visualisation

### 1. Electricity Prices in EU in Euros per KWH

In [54]:
def check_eu_prices():
    # Convert TIME_PERIOD as string
    eu_prices_df['TIME_PERIOD'] = eu_prices_df['TIME_PERIOD'].astype(str)

    # Sort by year and value
    eu_prices_df_sorted = eu_prices_df.sort_values(by=['TIME_PERIOD', 'OBS_VALUE'], ascending=[True, True])
    
    # Create bar plot
    fig = px.bar(
        data_frame=eu_prices_df_sorted,
        x='OBS_VALUE',
        y='country',
        orientation='h', 
        color='OBS_VALUE',  
        color_continuous_scale='Sunsetdark',
        animation_frame='TIME_PERIOD',  # Animate by year
        range_x=[eu_prices_df['OBS_VALUE'].min(), eu_prices_df['OBS_VALUE'].max()],
        width=600,   
        height=800,
          labels={
            'OBS_VALUE': '€ per kWh', 
            'TIME_PERIOD': 'Year',
            'country': 'Country'
        }
    )

    fig.update_layout(
        title=f'Electricity Prices by Country (EU)',
        xaxis_title='Price per kWh (Euro)',
        yaxis_title='Country',
        showlegend=True,
        template='plotly_white',
        paper_bgcolor='rgba(0,0,0,0)',  # Transparent outside the plot
        plot_bgcolor='rgba(0,0,0,0)'   
    )
    
    return fig

### 2. Indigenous Energy Production in IE

In [56]:
# Show a pie chart to compare different energy sources per year
def generate_pie_chart(data, start, end, columns, title): 
    
    # Filter data by year, providing start and end year
    filtered_data = ie_power_production_df.loc[(ie_power_production_df.Year > start) & (ie_power_production_df.Year<end)]
    
    # Group by Year
    yearly_data = filtered_data.groupby('Year')[columns].sum().reset_index()

    # Add the total renewable energy production for each year
    yearly_data['Total'] = yearly_data[columns].sum(axis=1)
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=len(yearly_data),
        specs=[[{'type':'domain'}]*len(yearly_data)],
        subplot_titles=[str(year) for year in yearly_data['Year']]
    )
    
    # Loop by yearly data
    for idx, row in yearly_data.iterrows():
        fig.add_trace(
            go.Pie(
                labels=columns,  # use correct column labels
                values=[row[col] for col in columns],  # pick values for that row
                name=str(row['Year']),
                hole=0.4,
                textinfo='label+percent',  
                hoverinfo='label+percent+value',  
                title=f"Total: {row['Total']:.2f} ktoe"
            ),
            row=1, 
            col=idx+1  # index is 0-based, subplot cols start at 1
        )
    # Update Titles 
    fig.update_layout(title_text=title, height=500)

    #Render Graph
    return fig

generate_pie_chart(ie_power_production_df, 2021, 2025, renewable_energy, 'Renewable Energy Production Sources for the Past 3 Years').show()

## 3. Get Renewable Enery Predictions

In [58]:
# Import Sarima Model
try: 
    with open('assets/model/sarima_model.pkl', 'rb') as f:
        model = pickle.load(f) 
except Exception as e:
    print(f'Error loading the model: {e}')

In [59]:
# Create a function to visualize renewable energy production in coming years
def show_power_predictions():

    # Forecast the next n steps
    n_steps = 60  # Prediction starts at the month of January 2022, add 12 steps per year
    forecast = model.get_forecast(steps=n_steps)
    mean_forecast = forecast.predicted_mean
    conf_int = forecast.conf_int()

    # Create figure
    fig = go.Figure()

    # Add actual data
    fig.add_trace(go.Scatter(
        x=ie_power_production_df.index,
        y=ie_power_production_df['Renewable_Energy'],
        mode='lines',
        name='Actual Outputs',
        line=dict(width=2)
    ))

    # Add forecasted mean
    fig.add_trace(go.Scatter(
        x=mean_forecast.index,
        y=mean_forecast,
        mode='lines',
        name='Forecasted Output',
        line=dict(width=2, color='green')
    ))

    # Add lower bound
    fig.add_trace(go.Scatter(
        x=conf_int.index,
        y=conf_int.iloc[:, 0],
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        name='Lower Bound',
        hoverinfo='skip'
    ))

    # Add upper bound and fill to previous
    fig.add_trace(go.Scatter(
        x=conf_int.index,
        y=conf_int.iloc[:, 1],
        fill='tonexty',
        fillcolor='rgba(255,0,0,0.1)',  # Semi-transparent red
        mode='lines',
        line=dict(width=0),
        name='Confidence Interval',
        hoverinfo='skip'
    ))

    # Update layout
    fig.update_layout(
        title='Actual vs Forecasted Renewable Energy',
        title_x=0.5,
        xaxis_title='Date',
        yaxis_title='Energy Output (ktoe)',
        template='plotly_white'
    )

    return fig

## Dashboard Implementation using Plotly Dash

Created an informative dashboard with the use of Plotly Dash and added figure charts comparison for Ireland's energy sources (renewable and non-renewable), European energy prices per kWh, and sentiment analysis results. <br />

Running the code will automatically open a new browser tab for a better user experience.

In [61]:
# Build Dash App
app = Dash(__name__)

app.layout = html.Div(style={
    'backgroundColor': '#f0f0f5',
    'minHeight': '100vh',
    'padding': '0',
    'fontFamily': 'Arial, sans-serif'
}, children=[

    # Header
    html.Div("Renewable Energy Forecast for the Republic of Ireland", style={
        'backgroundColor': 'rgb(80, 200, 120)',
        'color': 'white',
        'padding': '30px',
        'fontSize': '30px',
        'fontWeight': 'bold',
        'textAlign': 'center',
        'borderRadius': '10px',
        'boxShadow': '0 4px 8px rgba(0, 0, 0, 0.2)'
    }),

     # Prediction Container
        html.Div([
            dcc.Graph(figure=show_power_predictions(), config={
                'scrollZoom': False,'displayModeBar': True
            }, style={'height': '700px'})
        ], style={
            'width': '100%',
            'padding': '10px',
            'backgroundColor': 'white',
            'borderRadius': '10px',
            'boxShadow': '0 4px 8px rgba(0,0,0,0.1)',
            'verticalAlign': 'top'
        }),

    html.Div([
         # Prices Section
        html.Div([
            dcc.Graph(figure=check_eu_prices(), config={
                'scrollZoom': False,'displayModeBar': False
            }, style={'height': '700px'})
        ], style={
            'width': '32%',
            'padding': '10px',
            'backgroundColor': 'white',
            'borderRadius': '10px',
            'boxShadow': '0 4px 8px rgba(0,0,0,0.1)',
            'verticalAlign': 'top'
        }),

        # Energy Sources Section
        html.Div([
            ## This allows to insert the visualisation(graphs) into the divs

            # Show the renewable energy sources graph
            dcc.Graph(figure=generate_pie_chart(
                ie_power_production_df,
                2021, 2025,
                renewable_energy,
                'Renewable Energy Production Sources for the Past 3 Years'
            ),
            config={'scrollZoom': False, 'displayModeBar': True},
            style={'height': '450px'}),

            # Show the non-renewable energy sources graph
            dcc.Graph(figure=generate_pie_chart(
                ie_power_production_df,
                2021, 2025,
                nonrenewable_energy,
                'Non-renewable Energy Production Sources for the Past 3 Years'
            ), 
            config={'scrollZoom': False, 'displayModeBar': True},
            style={'height': '450px', 'marginTop': '0'})
        ], style={
            'width': '62%',
            'padding': '10px',
            'backgroundColor': 'white',
            'borderRadius': '10px',
            'boxShadow': '0 4px 8px rgba(0,0,0,0.1)',
            'display': 'flex',
            'flexDirection': 'column'
        }),


        # Sentiments Section
        html.Div([
            html.Div("Sentiments on Energy", style={
                'fontSize': '40px',
                'fontWeight': 'bold',
                'color': '#4CAF50',
                'marginBottom': '20px',
            }),
           html.Div([
                # Word Cloud 
                html.Img(
                    src='/assets/word_cloud.png',
                    style={
                        'width': '35%',
                        'borderRadius': '10px',
                        'marginRight': '20px'
                    }
                ),
            
                # Sentiment Plot 
                # This opens figures in html formats
                html.Iframe(
                    src='/assets/Sentiments_plot.html',
                    style={
                        'width': '55%',
                        'height': '550px',
                        'border': 'none'
                    }
                )
            ], style={
                'display': 'flex',
                'justifyContent': 'center',
                'alignItems': 'center',
                'gap': '20px',
                'padding': '30px',
                'flexWrap': 'wrap',
                'backgroundColor': 'white',
                'borderRadius': '10px'
            })
        ], style={
            'backgroundColor': 'white',
            'padding': '30px',
            'textAlign': 'center',
            'borderRadius': '10px',
            'boxShadow': '0 4px 8px rgba(0, 0, 0, 0.2)',
            'width': '100%',
            'display': 'flex',
            'flexDirection': 'column',
            'alignItems': 'center'
            
            
        })

    ], style={
        'width': '100%',
        'display': 'flex',
        'justifyContent': 'space-around',
        'flexWrap': 'wrap',
        'gap': '1rem',
        'marginTop': '20px'
    })
])

if __name__ == '__main__':
    app.run(debug=True)
    # Automatically open a new web browser tab
    webbrowser.open("http://127.0.0.1:8050")

## References

Plotly Technologies Inc., 2025. Dash in 20 Minutes Tutorial. Available at: https://dash.plotly.com/tutorial [Accessed 14 May 2025].

Importing Sarima Model:
Bais, G., 2025. How to Save Trained Model in Python. Neptune.ai. Available at: https://neptune.ai/blog/saving-trained-model-in-python [Accessed 14 May 2025].


Dataset: <br />
Anon, 2025. Electricity prices by type of user. Available at: http://data.europa.eu/88u/dataset/o0kgq8bifnbvc85xzncga [Accessed May 16, 2025].

Sustainable Energy Authority of Ireland (SEAI), 2024. Monthly Electricity Data – Customizable Download. [online] Available at: https://www.seai.ie/data-and-insights/seai-statistics/monthly-energy-data/electricity-monthly [Accessed 16 May 2025].
